# Representation Geometry & Information Analysis

**Experiment:** `TEST_ISAB5_NO_PROJECTOR`

Analyses:
1. Setup & Embedding Extraction
2. Covariance Spectral Analysis
3. Intrinsic Dimensionality
4. Isotropy & Norm Analysis
5. Non-Linear Probes (MLP vs Linear)
6. CKA (Centered Kernel Alignment)
7. Clustering & Discrete Structure
8. Information-Theoretic Measures
9. Summary Scorecard

## 1. Setup & Embedding Extraction

In [ ]:
import csv
import logging
import os
import sys
import warnings
from pathlib import Path

os.chdir(os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()))
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler, normalize as l2_normalize
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors
from rdkit import DataStructs

from utils.training import (
    load_config,
    build_model_from_config,
    load_pretrained_weights,
    latest_ckpt_path,
)
from input_pipeline import (
    TfLightningDataModule,
    _MASSSPEC_HF_REPO,
    _MASSSPEC_TSV_PATH,
    _download_hf_file,
    numpy_batch_to_torch,
)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)

In [ ]:
# Configuration
CHECKPOINT_DIR = "experiments/TEST_ISAB5_NO_PROJECTOR/trial_000"
CONFIG_PATH = "configs/gems_a_50_mask.py"
DEVICE = "cuda"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
rng = np.random.RandomState(RANDOM_SEED)

In [ ]:
# Load config and model
config = load_config(CONFIG_PATH)
datamodule = TfLightningDataModule(config, seed=int(config.seed))
config.num_peaks = datamodule.info["num_peaks"]
config.fingerprint_bits = int(datamodule.info["fingerprint_bits"])

backbone = build_model_from_config(config)
ckpt_path = latest_ckpt_path(Path(CHECKPOINT_DIR))
print(f"Loading checkpoint: {ckpt_path}")
load_pretrained_weights(backbone, ckpt_path)

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
backbone = backbone.to(device)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False

print(f"Model on {device}, params={sum(p.numel() for p in backbone.parameters()):,}")

In [ ]:
# Load SMILES from TSV
max_precursor_mz = float(config.get("max_precursor_mz", 1000.0))
tfrecord_dir = Path(config.get("tfrecord_dir", "data/gems_peaklist_tfrecord")).expanduser().resolve()
tsv_path = _download_hf_file(_MASSSPEC_HF_REPO, _MASSSPEC_TSV_PATH, tfrecord_dir.parent)

smiles_train = []
with Path(tsv_path).open() as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        if row["fold"] == "train" and float(row["precursor_mz"]) <= max_precursor_mz:
            smiles_train.append(row["smiles"])

print(f"SMILES for massspec_train: {len(smiles_train):,}")

In [ ]:
# Extract embeddings
probe_peak_ordering = str(config.peak_ordering)
dataset = datamodule.build_massspec_probe_dataset(
    "massspec_train",
    seed=RANDOM_SEED,
    peak_ordering=probe_peak_ordering,
    shuffle=False,
    drop_remainder=False,
)

embed_list = []
fp_list = []
meta = {"adduct": [], "instrument": [], "precursor_mz": [], "n_valid_peaks": []}

print("Extracting embeddings...")
with torch.no_grad():
    for numpy_batch in tqdm(dataset.as_numpy_iterator()):
        batch = numpy_batch_to_torch(numpy_batch)
        batch_dev = {k: v.to(device) for k, v in batch.items()}

        embeddings = backbone.encoder(
            batch_dev["peak_mz"],
            batch_dev["peak_intensity"],
            valid_mask=batch_dev["peak_valid_mask"],
            precursor_mz=batch_dev.get("precursor_mz"),
        )
        pooled = backbone.pool(embeddings, batch_dev["peak_valid_mask"])

        embed_list.append(pooled.cpu().float())
        fp_list.append(batch["fingerprint"].numpy())
        meta["adduct"].append(batch["adduct_id"].to(torch.long))
        meta["instrument"].append(batch["instrument_type_id"].to(torch.long))
        meta["precursor_mz"].append(batch["precursor_mz"].to(torch.float32))
        meta["n_valid_peaks"].append(batch["peak_valid_mask"].sum(dim=1).to(torch.long))

all_embeds = torch.cat(embed_list, dim=0).numpy()
all_fps_morgan = np.concatenate(fp_list, axis=0)
all_meta = {k: torch.cat(v).numpy() for k, v in meta.items()}
all_smiles = smiles_train[: len(all_embeds)]

print(f"Embeddings: {all_embeds.shape}")
print(f"Morgan FPs: {all_fps_morgan.shape}")
print(f"SMILES: {len(all_smiles)}")
assert len(all_smiles) == len(all_embeds)

In [ ]:
# RDKit feature extraction
n = len(all_smiles)
maccs_fps = np.zeros((n, 167), dtype=np.int8)

FG_SMARTS = {
    "hydroxyl": "[OX2H]",
    "carboxyl": "[CX3](=O)[OX2H1]",
    "amine": "[NX3;H2,H1;!$(NC=O)]",
    "amide": "[NX3][CX3](=[OX1])",
    "ester": "[#6][CX3](=O)[OX2H0][#6]",
    "ketone": "[#6][CX3](=O)[#6]",
    "aldehyde": "[CX3H1](=O)[#6]",
    "aromatic_ring": "c1ccccc1",
    "nitro": "[$([NX3](=O)=O),$([NX3+](=O)[O-])]",
    "sulfonyl": "[#16X4](=[OX1])(=[OX1])",
    "phosphate": "[PX4](=[OX1])([OX2])",
    "halide": "[F,Cl,Br,I]",
    "ether": "[OD2]([#6])[#6]",
    "thiol": "[#16X2H]",
    "nitrile": "[NX1]#[CX2]",
}
fg_patterns = {name: Chem.MolFromSmarts(sma) for name, sma in FG_SMARTS.items()}
fg_counts = {name: np.zeros(n, dtype=np.int16) for name in FG_SMARTS}

mol_weight = np.zeros(n, dtype=np.float32)
num_rings = np.zeros(n, dtype=np.int16)
num_aromatic_rings = np.zeros(n, dtype=np.int16)
num_heavy_atoms = np.zeros(n, dtype=np.int16)
logp = np.zeros(n, dtype=np.float32)
valid_mol_mask = np.ones(n, dtype=bool)

for i, smi in enumerate(tqdm(all_smiles, desc="Computing RDKit features")):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        valid_mol_mask[i] = False
        continue
    maccs = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros(167, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(maccs, arr)
    maccs_fps[i] = arr
    for name, pat in fg_patterns.items():
        fg_counts[name][i] = len(mol.GetSubstructMatches(pat))
    mol_weight[i] = Descriptors.ExactMolWt(mol)
    num_rings[i] = rdMolDescriptors.CalcNumRings(mol)
    num_aromatic_rings[i] = rdMolDescriptors.CalcNumAromaticRings(mol)
    num_heavy_atoms[i] = mol.GetNumHeavyAtoms()
    logp[i] = Descriptors.MolLogP(mol)

print(f"Valid molecules: {valid_mol_mask.sum():,} / {n:,}")

## 2. Covariance Spectral Analysis

In [ ]:
# SVD of centered embedding matrix
X_centered = all_embeds - all_embeds.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
eigenvalues = S ** 2 / (len(X_centered) - 1)  # variance along each PC

# Effective rank (participation ratio)
effective_rank = (S.sum()) ** 2 / (S ** 2).sum()

# Numerical rank at thresholds
cumvar = np.cumsum(eigenvalues) / eigenvalues.sum()
rank_90 = int(np.searchsorted(cumvar, 0.90)) + 1
rank_95 = int(np.searchsorted(cumvar, 0.95)) + 1
rank_99 = int(np.searchsorted(cumvar, 0.99)) + 1

# Condition number
condition_number = S[0] / S[-1] if S[-1] > 0 else float("inf")

print(f"Embedding dim: {all_embeds.shape[1]}")
print(f"Effective rank (participation ratio): {effective_rank:.2f}")
print(f"Numerical rank at 90% variance: {rank_90}")
print(f"Numerical rank at 95% variance: {rank_95}")
print(f"Numerical rank at 99% variance: {rank_99}")
print(f"Condition number (s[0]/s[-1]): {condition_number:.2f}")
print(f"Top-5 singular values: {S[:5]}")
print(f"Bottom-5 singular values: {S[-5:]}")

In [ ]:
# Eigenvalue spectrum plot
D = len(S)
uniform_eigenvalue = eigenvalues.sum() / D  # isotropic baseline

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Log-scale eigenvalue spectrum
axes[0].semilogy(range(1, D + 1), eigenvalues, "b-", lw=1.5, label="Observed")
axes[0].axhline(uniform_eigenvalue, color="r", ls="--", lw=1, label="Uniform (max isotropy)")
axes[0].set_xlabel("Component index")
axes[0].set_ylabel("Eigenvalue (log scale)")
axes[0].set_title("Eigenvalue Spectrum")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative explained variance
axes[1].plot(range(1, D + 1), cumvar, "b-", lw=1.5)
for thresh, rank, color in [(0.90, rank_90, "g"), (0.95, rank_95, "orange"), (0.99, rank_99, "r")]:
    axes[1].axhline(thresh, color=color, ls="--", lw=0.8, alpha=0.7)
    axes[1].axvline(rank, color=color, ls=":", lw=0.8, alpha=0.7)
    axes[1].annotate(f"{thresh:.0%}: d={rank}", xy=(rank, thresh), fontsize=9,
                     xytext=(rank + D * 0.05, thresh - 0.03))
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative explained variance")
axes[1].set_title("Explained Variance")
axes[1].grid(True, alpha=0.3)

# Eigenvalue ratio (log)
ratios = eigenvalues[:-1] / eigenvalues[1:]
axes[2].semilogy(range(1, len(ratios) + 1), ratios, "b-", lw=1)
axes[2].axhline(1.0, color="r", ls="--", lw=0.8)
axes[2].set_xlabel("Component index")
axes[2].set_ylabel("Eigenvalue ratio (i / i+1)")
axes[2].set_title("Eigenvalue Decay Rate")
axes[2].grid(True, alpha=0.3)

plt.suptitle("Covariance Spectral Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Intrinsic Dimensionality

In [ ]:
# TWO-NN estimator (Facco et al. 2017)
# ID = d such that mu = r2/r1 follows Pareto(d, 1)
# MLE: d = N / sum(log(mu_i))

id_subsample = min(20_000, len(all_embeds))
id_idx = rng.choice(len(all_embeds), size=id_subsample, replace=False)
X_id = all_embeds[id_idx]

nn = NearestNeighbors(n_neighbors=3, metric="euclidean", algorithm="brute", n_jobs=-1)
nn.fit(X_id)
dists, _ = nn.kneighbors(X_id)
r1 = dists[:, 1]  # distance to 1st NN (skip self)
r2 = dists[:, 2]  # distance to 2nd NN

# Filter out zero distances
valid = r1 > 1e-10
mu = r2[valid] / r1[valid]

# TWO-NN MLE
twonn_id = len(mu) / np.sum(np.log(mu))
print(f"TWO-NN intrinsic dimensionality: {twonn_id:.2f}")

In [ ]:
# MLE intrinsic dimensionality (Levina-Bickel 2004)
# For each point, estimate local ID from k nearest neighbors
K_VALUES_ID = [5, 10, 20, 50]
max_k = max(K_VALUES_ID) + 1

nn_lb = NearestNeighbors(n_neighbors=max_k, metric="euclidean", algorithm="brute", n_jobs=-1)
nn_lb.fit(X_id)
dists_lb, _ = nn_lb.kneighbors(X_id)

print("MLE intrinsic dimensionality (Levina-Bickel):")
mle_ids = {}
for k in K_VALUES_ID:
    # For each point, T_k = distance to k-th NN
    T_k = dists_lb[:, k]  # k-th neighbor (1-indexed since col 0 is self)
    # Local ID estimate: 1 / (1/(k-1) * sum_{j=1}^{k-1} log(T_k / T_j))
    log_ratios = np.zeros(len(X_id))
    for j in range(1, k):
        ratio = T_k / np.maximum(dists_lb[:, j], 1e-10)
        log_ratios += np.log(np.maximum(ratio, 1e-10))
    local_id = (k - 1) / log_ratios
    local_id = local_id[np.isfinite(local_id) & (local_id > 0) & (local_id < 1000)]
    mean_id = np.mean(local_id)
    median_id = np.median(local_id)
    mle_ids[k] = mean_id
    print(f"  k={k:3d}: mean={mean_id:.2f}, median={median_id:.2f}")

In [ ]:
# Local PCA dimensionality at different neighborhood sizes
LOCAL_PCA_K = [20, 50, 100, 200]
max_k_pca = max(LOCAL_PCA_K) + 1

# Subsample further for local PCA (expensive)
pca_subsample = min(5_000, id_subsample)
pca_idx = rng.choice(len(X_id), size=pca_subsample, replace=False)
X_pca = X_id  # full dataset for neighborhood lookup

nn_pca = NearestNeighbors(n_neighbors=max_k_pca, metric="euclidean", algorithm="brute", n_jobs=-1)
nn_pca.fit(X_pca)
_, nn_pca_idx = nn_pca.kneighbors(X_pca[pca_idx])

print("Local PCA dimensionality (90% variance threshold):")
local_pca_dims = {}
for k in LOCAL_PCA_K:
    dims = []
    for i in range(pca_subsample):
        neighbors = X_pca[nn_pca_idx[i, :k]]
        centered = neighbors - neighbors.mean(axis=0)
        _, s, _ = np.linalg.svd(centered, full_matrices=False)
        var_explained = np.cumsum(s ** 2) / (s ** 2).sum()
        local_dim = int(np.searchsorted(var_explained, 0.90)) + 1
        dims.append(local_dim)
    mean_dim = np.mean(dims)
    local_pca_dims[k] = mean_dim
    print(f"  k={k:4d}: mean local dim = {mean_dim:.1f}, median = {np.median(dims):.1f}")

In [ ]:
# Stratify TWO-NN ID by metadata
meta_id = {k: all_meta[k][id_idx] for k in ["precursor_mz", "n_valid_peaks"]}

print("\nTWO-NN ID stratified by precursor_mz bins:")
mz_bins = np.percentile(meta_id["precursor_mz"], [0, 25, 50, 75, 100])
for lo, hi in zip(mz_bins[:-1], mz_bins[1:]):
    mask = (meta_id["precursor_mz"] >= lo) & (meta_id["precursor_mz"] < hi + 1e-6)
    if mask.sum() < 100:
        continue
    X_sub = X_id[mask]
    nn_sub = NearestNeighbors(n_neighbors=3, metric="euclidean", algorithm="brute", n_jobs=-1)
    nn_sub.fit(X_sub)
    d_sub, _ = nn_sub.kneighbors(X_sub)
    r1_s, r2_s = d_sub[:, 1], d_sub[:, 2]
    v = r1_s > 1e-10
    mu_s = r2_s[v] / r1_s[v]
    id_s = len(mu_s) / np.sum(np.log(mu_s)) if len(mu_s) > 0 else float("nan")
    print(f"  mz [{lo:.0f}, {hi:.0f}): n={mask.sum():,}, ID={id_s:.2f}")

print("\nTWO-NN ID stratified by n_valid_peaks bins:")
peak_bins = np.percentile(meta_id["n_valid_peaks"], [0, 25, 50, 75, 100])
for lo, hi in zip(peak_bins[:-1], peak_bins[1:]):
    mask = (meta_id["n_valid_peaks"] >= lo) & (meta_id["n_valid_peaks"] < hi + 1)
    if mask.sum() < 100:
        continue
    X_sub = X_id[mask]
    nn_sub = NearestNeighbors(n_neighbors=3, metric="euclidean", algorithm="brute", n_jobs=-1)
    nn_sub.fit(X_sub)
    d_sub, _ = nn_sub.kneighbors(X_sub)
    r1_s, r2_s = d_sub[:, 1], d_sub[:, 2]
    v = r1_s > 1e-10
    mu_s = r2_s[v] / r1_s[v]
    id_s = len(mu_s) / np.sum(np.log(mu_s)) if len(mu_s) > 0 else float("nan")
    print(f"  peaks [{lo:.0f}, {hi:.0f}): n={mask.sum():,}, ID={id_s:.2f}")

## 4. Isotropy & Norm Analysis

In [ ]:
# Pairwise cosine similarity (subsample for feasibility)
iso_n = min(10_000, len(all_embeds))
iso_idx = rng.choice(len(all_embeds), size=iso_n, replace=False)
X_iso = l2_normalize(all_embeds[iso_idx], axis=1)

# Compute pairwise cosine similarities via dot product
cos_matrix = X_iso @ X_iso.T
# Extract upper triangle (excluding diagonal)
triu_idx = np.triu_indices(iso_n, k=1)
cos_values = cos_matrix[triu_idx]

isotropy_mean = cos_values.mean()
isotropy_std = cos_values.std()
# Isotropy score: 1 = perfectly isotropic (mean cos ~ 0), 0 = anisotropic
isotropy_score = 1.0 - abs(isotropy_mean)

print(f"Pairwise cosine similarity: mean={isotropy_mean:.6f}, std={isotropy_std:.6f}")
print(f"Isotropy score (1 - |mean_cos|): {isotropy_score:.6f}")

In [ ]:
# Embedding norm distribution
norms = np.linalg.norm(all_embeds, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Norm histogram
axes[0].hist(norms, bins=100, density=True, alpha=0.7, color="steelblue")
axes[0].axvline(norms.mean(), color="r", ls="--", label=f"mean={norms.mean():.2f}")
axes[0].set_xlabel("L2 Norm")
axes[0].set_ylabel("Density")
axes[0].set_title("Embedding Norm Distribution")
axes[0].legend()

print(f"Norm: mean={norms.mean():.4f}, std={norms.std():.4f}, "
      f"min={norms.min():.4f}, max={norms.max():.4f}, CV={norms.std()/norms.mean():.4f}")

# Norm vs precursor_mz
axes[1].scatter(all_meta["precursor_mz"], norms, s=0.5, alpha=0.1, rasterized=True)
r_mz = np.corrcoef(all_meta["precursor_mz"], norms)[0, 1]
axes[1].set_xlabel("Precursor m/z")
axes[1].set_ylabel("Embedding Norm")
axes[1].set_title(f"Norm vs Precursor m/z (r={r_mz:.4f})")

# Norm vs n_valid_peaks
axes[2].scatter(all_meta["n_valid_peaks"], norms, s=0.5, alpha=0.1, rasterized=True)
r_peaks = np.corrcoef(all_meta["n_valid_peaks"], norms)[0, 1]
axes[2].set_xlabel("Number of Valid Peaks")
axes[2].set_ylabel("Embedding Norm")
axes[2].set_title(f"Norm vs N Valid Peaks (r={r_peaks:.4f})")

plt.tight_layout()
plt.show()

# Norm correlations with molecular properties
valid = valid_mol_mask
print(f"\nNorm correlations (valid molecules only, n={valid.sum():,}):")
for name, vals in [("precursor_mz", all_meta["precursor_mz"]),
                   ("n_valid_peaks", all_meta["n_valid_peaks"]),
                   ("mol_weight", mol_weight),
                   ("logP", logp),
                   ("heavy_atoms", num_heavy_atoms)]:
    r = np.corrcoef(norms[valid], vals[valid])[0, 1]
    print(f"  {name:20s}: r={r:.4f}")

In [ ]:
# Per-dimension statistics
dim_means = all_embeds.mean(axis=0)
dim_vars = all_embeds.var(axis=0)
dim_kurtosis = stats.kurtosis(all_embeds, axis=0)  # excess kurtosis

print(f"Per-dimension variance: min={dim_vars.min():.6f}, max={dim_vars.max():.6f}, "
      f"ratio={dim_vars.max()/dim_vars.min():.2f}")
print(f"Per-dimension kurtosis: mean={dim_kurtosis.mean():.4f}, "
      f"min={dim_kurtosis.min():.4f}, max={dim_kurtosis.max():.4f}")

# Gaussianity test per dimension (Shapiro-Wilk on subsample)
sw_subsample = min(5000, len(all_embeds))  # Shapiro-Wilk max
sw_idx = rng.choice(len(all_embeds), size=sw_subsample, replace=False)
sw_pvalues = []
for d in range(all_embeds.shape[1]):
    _, p = stats.shapiro(all_embeds[sw_idx, d])
    sw_pvalues.append(p)
sw_pvalues = np.array(sw_pvalues)

n_gaussian = (sw_pvalues > 0.05).sum()
print(f"\nShapiro-Wilk normality test (n={sw_subsample}):")
print(f"  Gaussian dims (p > 0.05): {n_gaussian} / {all_embeds.shape[1]}")
print(f"  Median p-value: {np.median(sw_pvalues):.6f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(range(len(dim_vars)), np.sort(dim_vars)[::-1], color="steelblue", width=1.0)
axes[0].set_xlabel("Dimension (sorted)")
axes[0].set_ylabel("Variance")
axes[0].set_title("Per-Dimension Variance")

axes[1].hist(dim_kurtosis, bins=50, color="steelblue", alpha=0.7)
axes[1].axvline(0, color="r", ls="--", label="Gaussian (kurtosis=0)")
axes[1].set_xlabel("Excess Kurtosis")
axes[1].set_ylabel("Count")
axes[1].set_title("Per-Dimension Kurtosis")
axes[1].legend()

axes[2].hist(np.log10(sw_pvalues + 1e-50), bins=50, color="steelblue", alpha=0.7)
axes[2].axvline(np.log10(0.05), color="r", ls="--", label="p=0.05")
axes[2].set_xlabel("log10(p-value)")
axes[2].set_ylabel("Count")
axes[2].set_title("Shapiro-Wilk p-values")
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Non-Linear Probes (MLP vs Linear)

In [ ]:
# Subsample valid molecules for probing
valid_idx = np.where(valid_mol_mask)[0]
probe_size = min(30_000, len(valid_idx))
probe_idx = rng.choice(valid_idx, size=probe_size, replace=False)
X_probe = all_embeds[probe_idx]
X_probe_scaled = StandardScaler().fit_transform(X_probe)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Regression targets
reg_targets = {
    "mol_weight": mol_weight[probe_idx],
    "logP": logp[probe_idx],
    "heavy_atoms": num_heavy_atoms[probe_idx].astype(float),
    "num_rings": num_rings[probe_idx].astype(float),
}

# Classification targets (functional groups)
clf_targets = {}
for name in FG_SMARTS:
    y = (fg_counts[name][probe_idx] > 0).astype(int)
    pos_frac = y.mean()
    if 0.02 < pos_frac < 0.98:
        clf_targets[name] = y

print(f"Probe set: {probe_size:,} samples")
print(f"Regression targets: {list(reg_targets.keys())}")
print(f"Classification targets: {list(clf_targets.keys())}")

In [ ]:
# Regression probes: Linear vs MLP vs GradientBoosting
print("=== Regression Probes (R^2) ===")
print(f"{'Target':20s}  {'Ridge':>10s}  {'MLP':>10s}  {'GBT':>10s}  {'NL Gap':>10s}")
print("-" * 70)

reg_results = {}
for name, y in reg_targets.items():
    # Ridge (linear)
    ridge = Ridge(alpha=1.0)
    r2_ridge = cross_val_score(ridge, X_probe, y, cv=cv, scoring="r2", n_jobs=-1).mean()

    # MLP (2-layer)
    mlp = MLPRegressor(
        hidden_layer_sizes=(256, 128),
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=RANDOM_SEED,
    )
    r2_mlp = cross_val_score(mlp, X_probe_scaled, y, cv=cv, scoring="r2", n_jobs=-1).mean()

    # GradientBoosting
    gbt = GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.8,
        random_state=RANDOM_SEED,
    )
    r2_gbt = cross_val_score(gbt, X_probe, y, cv=cv, scoring="r2", n_jobs=-1).mean()

    best_nl = max(r2_mlp, r2_gbt)
    nl_gap = best_nl - r2_ridge
    reg_results[name] = {"ridge": r2_ridge, "mlp": r2_mlp, "gbt": r2_gbt, "nl_gap": nl_gap}

    print(f"{name:20s}  {r2_ridge:10.4f}  {r2_mlp:10.4f}  {r2_gbt:10.4f}  {nl_gap:+10.4f}")

In [ ]:
# Classification probes: LogReg vs MLP vs GradientBoosting
print("=== Classification Probes (AUC) ===")
print(f"{'Target':20s}  {'LogReg':>10s}  {'MLP':>10s}  {'GBT':>10s}  {'NL Gap':>10s}  {'Prev':>8s}")
print("-" * 80)

clf_results = {}
for name, y in clf_targets.items():
    prev = y.mean()

    # Logistic Regression (linear)
    lr = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs", n_jobs=-1)
    auc_lr = cross_val_score(lr, X_probe, y, cv=cv, scoring="roc_auc", n_jobs=-1).mean()

    # MLP
    mlp = MLPClassifier(
        hidden_layer_sizes=(256, 128),
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=RANDOM_SEED,
    )
    auc_mlp = cross_val_score(mlp, X_probe_scaled, y, cv=cv, scoring="roc_auc", n_jobs=-1).mean()

    # GradientBoosting
    gbt = GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.8,
        random_state=RANDOM_SEED,
    )
    auc_gbt = cross_val_score(gbt, X_probe, y, cv=cv, scoring="roc_auc", n_jobs=-1).mean()

    best_nl = max(auc_mlp, auc_gbt)
    nl_gap = best_nl - auc_lr
    clf_results[name] = {"logreg": auc_lr, "mlp": auc_mlp, "gbt": auc_gbt, "nl_gap": nl_gap}

    print(f"{name:20s}  {auc_lr:10.4f}  {auc_mlp:10.4f}  {auc_gbt:10.4f}  {nl_gap:+10.4f}  {prev:8.3f}")

## 6. CKA (Centered Kernel Alignment)

In [ ]:
def linear_cka(X: np.ndarray, Y: np.ndarray) -> float:
    """Linear CKA between two feature matrices (n_samples x n_features)."""
    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)
    # HSIC(K, L) = ||Y^T X||_F^2 / (n-1)^2
    XtY = X.T @ Y
    XtX = X.T @ X
    YtY = Y.T @ Y
    hsic_xy = np.sum(XtY ** 2)
    hsic_xx = np.sum(XtX ** 2)
    hsic_yy = np.sum(YtY ** 2)
    return float(hsic_xy / (np.sqrt(hsic_xx * hsic_yy) + 1e-10))


def rbf_cka(X: np.ndarray, Y: np.ndarray, sigma_x: float = None, sigma_y: float = None) -> float:
    """RBF CKA between two feature matrices."""
    n = X.shape[0]

    # Compute RBF kernels
    def rbf_kernel(Z, sigma):
        dists = pdist(Z, metric="sqeuclidean")
        if sigma is None:
            sigma = np.sqrt(np.median(dists))
        K = squareform(np.exp(-dists / (2 * sigma ** 2)))
        np.fill_diagonal(K, 1.0)
        return K, sigma

    Kx, sigma_x = rbf_kernel(X, sigma_x)
    Ky, sigma_y = rbf_kernel(Y, sigma_y)

    # Center kernels
    H = np.eye(n) - np.ones((n, n)) / n
    Kx_c = H @ Kx @ H
    Ky_c = H @ Ky @ H

    hsic_xy = np.sum(Kx_c * Ky_c)
    hsic_xx = np.sum(Kx_c * Kx_c)
    hsic_yy = np.sum(Ky_c * Ky_c)
    return float(hsic_xy / (np.sqrt(hsic_xx * hsic_yy) + 1e-10))

In [ ]:
# CKA computation
cka_n = min(10_000, len(valid_idx))
cka_idx = rng.choice(valid_idx, size=cka_n, replace=False)

X_cka = all_embeds[cka_idx]
FP_cka = all_fps_morgan[cka_idx].astype(np.float32)

# Molecular descriptor matrix
desc_cka = np.column_stack([
    mol_weight[cka_idx],
    logp[cka_idx],
    num_rings[cka_idx].astype(float),
    num_heavy_atoms[cka_idx].astype(float),
    all_meta["precursor_mz"][cka_idx],
])

# Linear CKA
lcka_embed_fp = linear_cka(X_cka, FP_cka)
lcka_embed_desc = linear_cka(X_cka, desc_cka)
lcka_fp_desc = linear_cka(FP_cka, desc_cka)

print("=== Linear CKA ===")
print(f"  Embedding vs Morgan FP:       {lcka_embed_fp:.4f}")
print(f"  Embedding vs Mol Descriptors: {lcka_embed_desc:.4f}")
print(f"  Morgan FP vs Mol Descriptors: {lcka_fp_desc:.4f}")

# RBF CKA (subsample further due to O(n^2) cost)
rbf_n = min(5_000, cka_n)
rbf_sub = rng.choice(cka_n, size=rbf_n, replace=False)

rbf_cka_embed_fp = rbf_cka(X_cka[rbf_sub], FP_cka[rbf_sub])
rbf_cka_embed_desc = rbf_cka(X_cka[rbf_sub], desc_cka[rbf_sub])

print("\n=== RBF CKA ===")
print(f"  Embedding vs Morgan FP:       {rbf_cka_embed_fp:.4f}")
print(f"  Embedding vs Mol Descriptors: {rbf_cka_embed_desc:.4f}")

## 7. Clustering & Discrete Structure

In [ ]:
# K-means clustering
clust_n = min(50_000, len(all_embeds))
clust_idx = rng.choice(len(all_embeds), size=clust_n, replace=False)
X_clust = all_embeds[clust_idx]

K_VALUES_CLUST = [10, 20, 50]
cluster_results = {}

print("=== K-Means Clustering ===")
print(f"{'k':>5s}  {'Calinski-H':>12s}  {'Davies-B':>10s}  {'Silhouette':>12s}")
print("-" * 50)

for k in K_VALUES_CLUST:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=3, max_iter=100)
    labels = km.fit_predict(X_clust)

    # Use subsample for silhouette (expensive)
    sil_n = min(10_000, clust_n)
    sil_sub = rng.choice(clust_n, size=sil_n, replace=False)

    ch = calinski_harabasz_score(X_clust, labels)
    db = davies_bouldin_score(X_clust, labels)
    sil = silhouette_score(X_clust[sil_sub], labels[sil_sub], metric="euclidean", sample_size=None)

    cluster_results[k] = {"labels": labels, "ch": ch, "db": db, "sil": sil}
    print(f"{k:5d}  {ch:12.1f}  {db:10.4f}  {sil:12.4f}")

In [ ]:
# Cluster purity and ARI/NMI vs metadata
clust_meta = {k: all_meta[k][clust_idx] for k in ["adduct", "instrument"]}

# Also get functional group labels for clustering analysis
# Use presence of most common functional groups
clust_fg = {name: (fg_counts[name][clust_idx] > 0).astype(int) for name in clf_targets}

print("\n=== Cluster Purity vs Metadata ===")
for k in K_VALUES_CLUST:
    labels = cluster_results[k]["labels"]
    print(f"\nk={k}:")

    # ARI and NMI vs adduct and instrument
    for meta_name in ["adduct", "instrument"]:
        meta_labels = clust_meta[meta_name]
        ari = adjusted_rand_score(meta_labels, labels)
        nmi = normalized_mutual_info_score(meta_labels, labels)
        print(f"  vs {meta_name:12s}: ARI={ari:.4f}, NMI={nmi:.4f}")

    # ARI/NMI vs functional groups (binarized)
    for fg_name, fg_labels in list(clust_fg.items())[:5]:  # top 5
        ari = adjusted_rand_score(fg_labels, labels)
        nmi = normalized_mutual_info_score(fg_labels, labels)
        print(f"  vs fg_{fg_name:12s}: ARI={ari:.4f}, NMI={nmi:.4f}")

In [ ]:
# Fisher Discriminant Ratio for adduct and instrument classes
print("\n=== Fisher Discriminant Ratio (class separability) ===")

for meta_name in ["adduct", "instrument"]:
    meta_labels = all_meta[meta_name]
    unique_labels = np.unique(meta_labels)
    # Filter classes with enough samples
    class_counts = {c: (meta_labels == c).sum() for c in unique_labels}
    valid_classes = [c for c, cnt in class_counts.items() if cnt >= 100]

    if len(valid_classes) < 2:
        print(f"  {meta_name}: insufficient classes")
        continue

    global_mean = all_embeds.mean(axis=0)
    # Between-class scatter
    S_b = np.zeros((all_embeds.shape[1], all_embeds.shape[1]))
    S_w = np.zeros_like(S_b)

    for c in valid_classes:
        mask = meta_labels == c
        X_c = all_embeds[mask]
        n_c = len(X_c)
        mean_c = X_c.mean(axis=0)
        diff = (mean_c - global_mean).reshape(-1, 1)
        S_b += n_c * (diff @ diff.T)
        X_c_centered = X_c - mean_c
        S_w += X_c_centered.T @ X_c_centered

    # Fisher criterion: trace(S_b) / trace(S_w)
    fisher_ratio = np.trace(S_b) / (np.trace(S_w) + 1e-10)
    print(f"  {meta_name:12s}: Fisher ratio = {fisher_ratio:.6f} "
          f"({len(valid_classes)} classes with >=100 samples)")

## 8. Information-Theoretic Measures

In [ ]:
# Mutual information between embedding PCs and metadata
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

# Project embeddings to top PCs
n_pcs_mi = min(50, all_embeds.shape[1])
mi_n = min(20_000, len(valid_idx))
mi_idx = rng.choice(valid_idx, size=mi_n, replace=False)

pca_mi = PCA(n_components=n_pcs_mi, random_state=RANDOM_SEED)
X_pcs = pca_mi.fit_transform(all_embeds[mi_idx])

print("=== Mutual Information: Embedding PCs vs Metadata ===")
print(f"Using top {n_pcs_mi} PCs, {mi_n:,} samples\n")

# Continuous targets
cont_targets_mi = {
    "precursor_mz": all_meta["precursor_mz"][mi_idx],
    "mol_weight": mol_weight[mi_idx],
    "logP": logp[mi_idx],
    "heavy_atoms": num_heavy_atoms[mi_idx].astype(float),
    "num_rings": num_rings[mi_idx].astype(float),
}

print("Continuous targets (sum MI across PCs):")
mi_cont_results = {}
for name, y in cont_targets_mi.items():
    mi_values = mutual_info_regression(X_pcs, y, random_state=RANDOM_SEED, n_neighbors=5)
    total_mi = mi_values.sum()
    top5_mi = np.sort(mi_values)[-5:].sum()
    mi_cont_results[name] = total_mi
    print(f"  {name:20s}: total MI={total_mi:.4f}, top-5 PCs MI={top5_mi:.4f}")

# Discrete targets
disc_targets_mi = {
    "adduct": all_meta["adduct"][mi_idx],
    "instrument": all_meta["instrument"][mi_idx],
}

print("\nDiscrete targets (sum MI across PCs):")
mi_disc_results = {}
for name, y in disc_targets_mi.items():
    mi_values = mutual_info_classif(X_pcs, y, random_state=RANDOM_SEED, n_neighbors=5)
    total_mi = mi_values.sum()
    mi_disc_results[name] = total_mi
    print(f"  {name:20s}: total MI={total_mi:.4f}")

In [ ]:
# Conditional entropy of metadata given embedding neighborhoods
# H(Y|X_nn) = avg entropy of label distribution among k-NN

ce_n = min(15_000, len(all_embeds))
ce_idx = rng.choice(len(all_embeds), size=ce_n, replace=False)
X_ce = all_embeds[ce_idx]

k_ce = 20
nn_ce = NearestNeighbors(n_neighbors=k_ce + 1, metric="euclidean", algorithm="brute", n_jobs=-1)
nn_ce.fit(X_ce)
_, nn_ce_idx = nn_ce.kneighbors(X_ce)
nn_ce_idx = nn_ce_idx[:, 1:]  # exclude self

print(f"=== Conditional Entropy H(Y | k={k_ce} NN) ===")

for meta_name in ["adduct", "instrument"]:
    labels = all_meta[meta_name][ce_idx]
    unique = np.unique(labels)
    n_classes = len(unique)

    # Unconditional entropy
    counts = np.bincount(labels.astype(int))
    probs = counts[counts > 0] / counts.sum()
    h_uncond = -np.sum(probs * np.log2(probs + 1e-10))

    # Conditional entropy: average entropy in neighborhoods
    h_cond = 0.0
    for i in range(ce_n):
        nn_labels = labels[nn_ce_idx[i]]
        nn_counts = np.bincount(nn_labels.astype(int), minlength=int(unique.max()) + 1)
        nn_probs = nn_counts[nn_counts > 0] / nn_counts.sum()
        h_cond += -np.sum(nn_probs * np.log2(nn_probs + 1e-10))
    h_cond /= ce_n

    mi_estimate = h_uncond - h_cond
    print(f"  {meta_name:12s}: H(Y)={h_uncond:.4f} bits, H(Y|NN)={h_cond:.4f} bits, "
          f"MI={mi_estimate:.4f} bits, reduction={mi_estimate/h_uncond:.1%}")

In [ ]:
# KL divergence of per-class embedding distributions from global
# Approximate using Gaussian fit per class

print("\n=== KL Divergence (per-class vs global, Gaussian approx) ===")

# Use PCA-reduced embeddings for stable covariance estimation
n_pcs_kl = min(30, all_embeds.shape[1])
pca_kl = PCA(n_components=n_pcs_kl, random_state=RANDOM_SEED)
X_kl = pca_kl.fit_transform(all_embeds)

global_mean = X_kl.mean(axis=0)
global_cov = np.cov(X_kl, rowvar=False) + 1e-6 * np.eye(n_pcs_kl)
global_cov_inv = np.linalg.inv(global_cov)
global_logdet = np.linalg.slogdet(global_cov)[1]

for meta_name in ["adduct", "instrument"]:
    labels = all_meta[meta_name]
    unique = np.unique(labels)
    print(f"\n  {meta_name} ({len(unique)} classes):")

    kl_values = []
    for c in unique:
        mask = labels == c
        if mask.sum() < n_pcs_kl + 10:
            continue
        X_c = X_kl[mask]
        mean_c = X_c.mean(axis=0)
        cov_c = np.cov(X_c, rowvar=False) + 1e-6 * np.eye(n_pcs_kl)

        # KL(class || global) for multivariate Gaussians
        cov_c_inv = np.linalg.inv(cov_c)
        logdet_c = np.linalg.slogdet(cov_c)[1]
        diff = mean_c - global_mean
        kl = 0.5 * (
            np.trace(global_cov_inv @ cov_c)
            + diff @ global_cov_inv @ diff
            - n_pcs_kl
            + global_logdet
            - logdet_c
        )
        kl_values.append((int(c), mask.sum(), float(kl)))

    kl_values.sort(key=lambda x: -x[2])
    for c, cnt, kl in kl_values[:10]:  # top 10
        print(f"    class {c:3d} (n={cnt:6,}): KL={kl:.4f} nats")
    if len(kl_values) > 10:
        kl_all = [x[2] for x in kl_values]
        print(f"    ... mean KL={np.mean(kl_all):.4f}, max={max(kl_all):.4f}")

## 9. Summary Scorecard

In [ ]:
# Collect all metrics into a summary
print("=" * 80)
print("REPRESENTATION ANALYSIS SCORECARD")
print(f"Experiment: {CHECKPOINT_DIR}")
print(f"Checkpoint: {ckpt_path}")
print(f"Embeddings: {all_embeds.shape}")
print("=" * 80)

print("\n--- Spectral Properties ---")
print(f"{'Effective rank':40s}: {effective_rank:.2f}")
print(f"{'Rank @90% variance':40s}: {rank_90}")
print(f"{'Rank @95% variance':40s}: {rank_95}")
print(f"{'Rank @99% variance':40s}: {rank_99}")
print(f"{'Condition number':40s}: {condition_number:.2f}")

print("\n--- Intrinsic Dimensionality ---")
print(f"{'TWO-NN ID':40s}: {twonn_id:.2f}")
for k, v in mle_ids.items():
    print(f"{'MLE ID (k=' + str(k) + ')':40s}: {v:.2f}")
for k, v in local_pca_dims.items():
    print(f"{'Local PCA dim (k=' + str(k) + ')':40s}: {v:.1f}")

print("\n--- Isotropy & Norms ---")
print(f"{'Mean pairwise cosine sim':40s}: {isotropy_mean:.6f}")
print(f"{'Std pairwise cosine sim':40s}: {isotropy_std:.6f}")
print(f"{'Isotropy score (1-|mean_cos|)':40s}: {isotropy_score:.6f}")
print(f"{'Norm mean':40s}: {norms.mean():.4f}")
print(f"{'Norm std':40s}: {norms.std():.4f}")
print(f"{'Norm CV':40s}: {norms.std()/norms.mean():.4f}")
print(f"{'Gaussian dims (Shapiro p>0.05)':40s}: {n_gaussian} / {all_embeds.shape[1]}")
print(f"{'Mean excess kurtosis':40s}: {dim_kurtosis.mean():.4f}")

print("\n--- Regression Probes (R^2) ---")
print(f"{'Target':20s}  {'Ridge':>8s}  {'MLP':>8s}  {'GBT':>8s}  {'NL Gap':>8s}")
for name, res in reg_results.items():
    print(f"{name:20s}  {res['ridge']:8.4f}  {res['mlp']:8.4f}  {res['gbt']:8.4f}  {res['nl_gap']:+8.4f}")

print("\n--- Classification Probes (AUC) ---")
print(f"{'Target':20s}  {'LogReg':>8s}  {'MLP':>8s}  {'GBT':>8s}  {'NL Gap':>8s}")
for name, res in clf_results.items():
    print(f"{name:20s}  {res['logreg']:8.4f}  {res['mlp']:8.4f}  {res['gbt']:8.4f}  {res['nl_gap']:+8.4f}")

print("\n--- CKA ---")
print(f"{'Linear CKA: Embed vs Morgan FP':40s}: {lcka_embed_fp:.4f}")
print(f"{'Linear CKA: Embed vs Mol Desc':40s}: {lcka_embed_desc:.4f}")
print(f"{'RBF CKA: Embed vs Morgan FP':40s}: {rbf_cka_embed_fp:.4f}")
print(f"{'RBF CKA: Embed vs Mol Desc':40s}: {rbf_cka_embed_desc:.4f}")

print("\n--- Clustering (k=20) ---")
k20 = cluster_results[20]
print(f"{'Calinski-Harabasz':40s}: {k20['ch']:.1f}")
print(f"{'Davies-Bouldin':40s}: {k20['db']:.4f}")
print(f"{'Silhouette':40s}: {k20['sil']:.4f}")

print("\n--- Mutual Information (sum over PCs) ---")
for name, v in mi_cont_results.items():
    print(f"{'MI: ' + name:40s}: {v:.4f}")
for name, v in mi_disc_results.items():
    print(f"{'MI: ' + name:40s}: {v:.4f}")

print("\n" + "=" * 80)